---
authors: Freek Pols
---
# Brownian motion

Een van de bekendste voorbeelden van botsende deeltjes in de natuur is Brownian motion.
Fijn gemalen pollen in water lijken te dansen in willekeurige richting.
Dit komt doordat de pollen worden geraakt door watermoleculen die in alle richtingen bewegen.
Omdat de pollen veel zwaarder zijn dan watermoleculen, dus de beweging van de pollen is veel langzamer en minder "intens" dan die van de watermoleculen.
Dit proces van willekeurige beweging door botsingen met kleinere deeltjes wordt Brownian motion genoemd en kunnen we simuleren op basis van ons (premature) botsingsmodel.
Daarbij kunnen we ook gebruik maken van de zojuist geleerde manier van tracking van deeltjes, waarbij we een zowel het zware bolletjes als een enkel deeltje kunnen volgen.

Let op!
We bestuderen hier nog geen thermische effecten, deze opdrachten zijn met name bedoeld om beter te begrijpen hoe het botsingsmodel in elkaar zit.

```{warning}
In dit notebook zitten delen waar ruimte is om code toe te voegen, maar waarbij je denkt... waarom dan?
In een latere opdracht moet je terug naar die cell en de juiste code toevoegen.
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Maken van de ParticleClass

class ParticleClass:
    # Het maken van het deeltje
    def __init__(self, m, v, r, R, c):
        self.m = m                         
        self.v = np.array(v, dtype=float)  
        self.r = np.array(r, dtype=float)  
        self.R = np.array(R, dtype=float)  
        self.c = c

    # Het updaten van de positie, eventueel met zwaartekracht
    def update_position(self):
        self.r += self.v * dt #+ 1/2 * a * dt**2  
              
    # Harde wand
    def boxcollision(self):
        if abs(self.r[0]) + self.R > Box_length: 
            self.v[0] = -self.v[0]                                  # Omdraaien van de snelheid
            self.r[0] = np.sign(self.r[0]) * (Box_length - self.R)  # Zet terug net binnen box                 
        if abs(self.r[1]) + self.R > Box_length: 
            self.v[1] = -self.v[1]     
            self.r[1] = np.sign(self.r[1]) * (Box_length - self.R) 
            
    @property
    def momentum(self):
        return self.m * self.v
    
    @property
    def kin_energy(self):
        return 1/2 * self.m * np.dot(self.v, self.v)

In [ ]:
# Aanmaken van de randvoorwaarden en initiele condities
Box_size_0 = 10
Box_length_0 = Box_size_0/2
Box_length = Box_length_0     # De grootte van de box kan wijzigen!

# Particles
dt = 0.1
particles = []
N = 40
v_0 = 1

dt = 0.04

# Aanmaken van deeltjes
for i in range(N-1):
    vx = np.random.uniform(-v_0,v_0)
    vy = np.random.choice([-1, 1])*np.sqrt(v_0**2-vx**2)        
    pos = Box_length_0*np.random.uniform(-1,1,2)
    particles.append(ParticleClass(m=1.0, v=[vx, vy], r = pos, R=.5,c='blue')) 

particles.append(ParticleClass(m=20.0, v=[0, 0], r = [0, 0], R=.5,c='red')) 


```{exercise} Brownian motion
:label: ex-brownian-1

Leg uit wat er in de laatste regel van bovenstaande script gebeurt.
Voeg voor deze regel goede metadata toe.
```

```{solution} ex-brownian-1

# Uitleg:
# De laatste regel maakt het 'zware' deeltje aan (het pollenkorrel).
# Het heeft een massa van 20 (veel zwaarder dan de 1.0 van de andere deeltjes),
# begint met snelheid 0 in het midden [0,0] en is rood gekleurd.

# Metadata:
# particles.append(ParticleClass(m=20.0, v=[0, 0], r = [0, 0], R=.5,c='red')) 
```

Er is een doos vol met deeltjes op willekeurige positie aangemaakt.
We willen kijken waar de deeltjes zijn terechtgekomen.
Hieronder staat dit weergegeven. 



In [ ]:
# Inspecteren van beginsituatie
plt.figure()

plt.xlabel('x')
plt.ylabel('y')

plt.xlim(-Box_length_0,Box_length_0)
plt.ylim(-Box_length_0,Box_length_0)


for particle, particle_object in enumerate(particles):
    plt.plot(particle_object.r[0],particle_object.r[1],color=particle_object.c,marker='.')
    # plt.arrow(particle_object.r[0],particle_object.r[1], 
    #           particle_object.v[0],particle_object.v[1], 
    #           head_width=0.05, head_length=0.1, color='red')
plt.show()


```{exercise} 
:label: ex-brownian-2

Er staat ook code met comments ervoor, wat doet deze code? 
Check je antwoord door de comments weg te halen.
Hoe wordt er voor gezorgd dat de snelheid van elk deeltje gelijk is?
```

```{solution} ex-brownian-2

# 1. De code plt.arrow(...) tekent een pijl (vector) die de snelheid en richting
#    van het deeltje visualiseert.

# 2. Gelijkblijvende snelheid:
#    Bij de initialisatie wordt vx willekeurig gekozen.
#    vy wordt berekend met Pythagoras: vy = sqrt(v0^2 - vx^2).
#    Hierdoor is de totale snelheid (lengte van de vector) altijd v0.
```

We gaan nu de functies van de simulatie weer aanroepen:

In [ ]:
# Het bepalen of er een botsing plaats vindt
def collide_detection(self, other):
    dx = self.r[0] - other.r[0]
    dy = self.r[1] - other.r[1]
    rr = self.R + other.R
    return  dx**2+dy**2 < rr**2 
        
def particle_collision(p1: ParticleClass, p2: ParticleClass):
    """ past snelheden aan uitgaande van overlap """
    m1, m2 = p1.m, p2.m
    delta_r = p1.r - p2.r
    delta_v = p1.v - p2.v
    dot_product = np.dot(delta_r, delta_v)
    
    # Als deeltjes van elkaar weg bewegen dan geen botsing
    if dot_product >= 0: # '='-teken voorkomt ook problemen als delta_r == \vec{0}
        return
    
    distance_squared = np.dot(delta_r, delta_r) 
    # Botsing oplossen volgens elastische botsing in 2D
    p1.v -= 2 * m2 / (m1 + m2) * dot_product / distance_squared * delta_r
    p2.v += 2 * m1 / (m1 + m2) * dot_product / distance_squared * delta_r

def handle_collisions(particles):
    """ alle onderlinge botsingen afhandelen voor deeltjes in lijst """
    num_particles = len(particles)
    for i in range(num_particles):
        for j in range(i+1, num_particles):
            if collide_detection(particles[i], particles[j]):
                particle_collision(particles[i], particles[j])


In onderstaande code geven we de code voor de simulatie en volgen we de positie van het zware deeltje. 

In [ ]:
# tracken van het zware deeltje
track_x = []
track_y = []

# tracken van een licht deeltje (het eerste deeltje in de lijst)
track_x_light = []
track_y_light = []

for i in range(400):
    
    for p in particles:
        p.update_position()    # Update positie        
        p.boxcollision()         # Wandbotsing werkt per deeltje
        
    handle_collisions(particles)
    
    track_x.append(particles[N-1].r[0])
    track_y.append(particles[N-1].r[1])
    
    # Tracking data voor licht deeltje
    track_x_light.append(particles[0].r[0])
    track_y_light.append(particles[0].r[1])

plt.figure(figsize=(8,8))
plt.title("Brownian Motion: Heavy (Red) vs Light (Blue) Particle")
plt.xlabel("x")
plt.ylabel("y")
plt.xlim(-Box_length_0, Box_length_0)
plt.ylim(-Box_length_0, Box_length_0)

# Plot pad zwaar deeltje
plt.plot(track_x, track_y, 'r-', linewidth=2, label='Heavy Particle')

# Plot pad licht deeltje
plt.plot(track_x_light, track_y_light, 'c--', linewidth=1, alpha=0.7, label='Light Particle')

plt.legend()
plt.show()

```{exercise} Brownian motion in beeld
:label: ex-brownian-3
- Draai de onderstaande simulatie een keer en bestudeer de output.
- Voeg zelf een tweede tracking toe van een licht deeltje en verbeter de plot.
- Wat zijn overeenkomsten en verschillen tussen de beweging van de twee deeltjes?
- Wat valt je op als je de simulatie een aantal keer runt?
```

```{solution} ex-brownian-3
1. Verschil: Het lichte deeltje beweegt zeer snel door de hele box en botst vaak tegen wanden.
   Het zware deeltje beweegt langzaam en blijft in een kleiner gebied (Brownian motion).
2. Observatie: Het pad van het zware deeltje is 'jagged' (rafelig) door de vele kleine botsingen,
   maar heeft veel meer inertie.
```

We zouden gevoel willen krijgen voor het aantal botsingen dat per tijdseenheid plaatsvindt. 
Elke keer dat er een botsing plaatsvindt, zou de counter met 1 omhoog moeten gaan.
Idealiter wordt het aantal botsingen opgeslagen in een array zodat je het aantal botsingen als functie van de tijd kunt weergeven.

```{exercise}
Pas bovenstaand idee toe in de eerder gemaakte code.
Plot hieronder het aantal botsingen als functie van de tijd.
```

In [ ]:
# Modified collision handler that returns the count of collisions
def handle_collisions_with_count(particles):
    collisions = 0
    num_particles = len(particles)
    for i in range(num_particles):
        for j in range(i+1, num_particles):
            if collide_detection(particles[i], particles[j]):
                particle_collision(particles[i], particles[j])
                collisions += 1
    return collisions

# Reset particles
particles = []
Box_length = Box_length_0

# Create particles again for clean run
for i in range(N-1):
    vx = np.random.uniform(-v_0,v_0)
    vy = np.random.choice([-1, 1])*np.sqrt(v_0**2-vx**2)        
    pos = Box_length_0*np.random.uniform(-1,1,2)
    particles.append(ParticleClass(m=1.0, v=[vx, vy], r = pos, R=.5,c='blue')) 
particles.append(ParticleClass(m=20.0, v=[0, 0], r = [0, 0], R=.5,c='red'))

# Data storage
collision_history = []
time_steps = []

# Run Simulation
for i in range(400):
    for p in particles:
        p.update_position()      
        p.boxcollision()         
    
    # Get number of collisions in this step
    count = handle_collisions_with_count(particles)
    collision_history.append(count)
    time_steps.append(i * dt)

# Plot
plt.figure(figsize=(10,4))
plt.plot(time_steps, collision_history)
plt.xlabel('Time (s)')
plt.ylabel('Number of Collisions')
plt.title('Collisions per Time Step')
plt.show()

```{warning} 🌶 Let op!
:icon: false
De onderstaande opdrachten vallen buiten de stof maar tellen mee als je excellent wilt behalen.
```

In zulke fysica modellen is de afgelegde weg (afstand tussen begin en eindpunt) van belang.
Deze afgelegde weg zegt iets over de snelheid van difussie.
Idealiter bekijken we een histogram.
Maar voor een histogram hebben we veel deeltjes nodig.



```{exercise} Afgelegde weg 🌶
:label: ex-brownian-4

- Maak een simulatie met 361 deeltjes, waarvan 1 zwaar deeltje.
- Houd rekening met de boxgrootte, deze moet mee schalen!
- Maak een histogram van de afgelegde weg voor alle deeltjes. 
- Geef de afgelegde weg van het grote deeltje duidelijk aan.
```


In [ ]:
# 1. Configuration
N_total = 361
N_light = N_total - 1
Box_size_new = 30 # Scaled up box
Box_length = Box_size_new / 2
v_0 = 1
dt = 0.04

# 2. Initialize Particles
particles = []

# Light particles
for i in range(N_light):
    vx = np.random.uniform(-v_0, v_0)
    vy = np.random.choice([-1, 1]) * np.sqrt(v_0**2 - vx**2)
    pos = Box_length * np.random.uniform(-0.9, 0.9, 2) 
    particles.append(ParticleClass(m=1.0, v=[vx, vy], r=pos, R=0.5, c='blue'))

# Heavy particle (at center)
particles.append(ParticleClass(m=20.0, v=[0, 0], r=[0, 0], R=0.5, c='red'))

# Store initial positions
start_positions = [np.copy(p.r) for p in particles]

# 3. Run Simulation
steps = 400
for step in range(steps):
    for p in particles:
        p.update_position()
        p.boxcollision()
    handle_collisions(particles)

# 4. Calculate Displacements
displacements = []
for i, p in enumerate(particles):
    dist = np.linalg.norm(p.r - start_positions[i])
    displacements.append(dist)

# 5. Plot Histogram
plt.figure(figsize=(10, 6))
plt.hist(displacements[:-1], bins=20, alpha=0.7, label='Light Particles')
plt.axvline(displacements[-1], color='red', linestyle='dashed', linewidth=2, label='Heavy Particle')
plt.xlabel('Displacement (Afgelegde weg)')
plt.ylabel('Count')
plt.title('Histogram of Particle Displacements')
plt.legend()
plt.show()

En nu we toch bezig zijn met twee verschillende deeltjes.... 

We kunnen twee "groepen" van deeltjes aanmaken, elk  met een andere massa. Als we dan de zwaartekracht aan zetten, dan zouden we verwachten dat de lichtere deeltjes boven komen "drijven".

```{exercise} Onderzoek dit vermoeden 🌶
- maak daartoe de box 2x zo hoog als breed
- verdubbel het totaal aantal deeltjes
- zet een artificieel grote zwaartekracht aan
```

In [ ]:
# 1. Update Particle Class to include Gravity
class ParticleClassGravity(ParticleClass):
    def update_position(self):
        g = 0.5 # Gravity strength
        self.v[1] -= g * dt
        self.r += self.v * dt

# 2. Setup Rectangular Box
Box_width = 10
Box_height = 20
Lx = Box_width / 2
Ly = Box_height / 2

# Custom rectangular collision
def boxcollision_rect(p, Lx, Ly):
    if abs(p.r[0]) + p.R > Lx: 
        p.v[0] = -p.v[0]
        p.r[0] = np.sign(p.r[0]) * (Lx - p.R)
    if abs(p.r[1]) + p.R > Ly: 
        p.v[1] = -p.v[1]     
        p.r[1] = np.sign(p.r[1]) * (Ly - p.R)

# 3. Initialize Particles
particles = []
N = 80
for i in range(N):
    mass = np.random.choice([1.0, 10.0]) # Random mass
    color = 'blue' if mass == 1.0 else 'red'
    
    vx = np.random.uniform(-1, 1)
    vy = np.random.uniform(-1, 1)
    pos_x = (Lx - 1) * np.random.uniform(-1, 1)
    pos_y = (Ly - 1) * np.random.uniform(-1, 1)
    
    particles.append(ParticleClassGravity(m=mass, v=[vx, vy], r=[pos_x, pos_y], R=0.5, c=color))

# 4. Run Simulation
steps = 600
for step in range(steps):
    for p in particles:
        p.update_position()
        boxcollision_rect(p, Lx, Ly)
    handle_collisions(particles)

# 5. Plot Final State
plt.figure(figsize=(5, 10))
plt.title("Gravity Effect: Heavy (Red) vs Light (Blue)")
plt.xlim(-Lx, Lx)
plt.ylim(-Ly, Ly)
for p in particles:
    plt.plot(p.r[0], p.r[1], marker='o', color=p.c, markersize=8 if p.m > 1 else 4)
plt.show()